# Inverse dynamica van een vereenvoudigd vijfstangenmechanisme

Deze notebook gebruikt bewust een eenvoudiger mechanisme dan het volledige ruitenwissermechanisme: een vlakke serieketen met vijf stangen en vijf draaigewrichten. Dat is geen exacte kopie van de ruitenwisser, maar wel een duidelijk model om inverse dynamica met zwaartekracht en wrijving te tonen.

Voor het volledige ruitenwissermechanisme zouden zwaartekracht en wrijving ook kunnen worden toegevoegd, maar dan worden alle lusvergelijkingen, reactiekrachten en tekenconventies meteen veel moeilijker te controleren. Voor de opdracht is dit vereenvoudigde vijfstangenmodel nuttiger: het toont dezelfde fysica, met overzichtelijke vergelijkingen.

De analyse berekent:

- massacentrumposities, snelheden en versnellingen;
- gewrichtskrachten;
- aandrijfkoppels zonder zwaartekracht/wrijving;
- aandrijfkoppels met zwaartekracht;
- aandrijfkoppels met zwaartekracht en gewrichtswrijving;
- mechanisch vermogen en motorselectie.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## Model en parameters

Het model is een open 5R-keten. Elke stang heeft een lengte `L[i]`, massa `m[i]`, massacentrum op `c[i] = L[i]/2`, en traagheidsmoment rond het massacentrum.

De wrijving in elk gewricht wordt gemodelleerd als:

\[
M_{f,i} = b_i \dot q_i + M_{c,i}\tanh\left(\frac{\dot q_i}{v_\epsilon}\right)
\]

waarbij `b_i` viscose wrijving is en `M_c,i` een gladde benadering van Coulombwrijving.

In [ ]:
# Geometry and mass properties
n_links = 5
L = np.array([0.28, 0.24, 0.20, 0.17, 0.14])       # link lengths [m]
c = 0.5 * L                                        # center of mass distances [m]
m = np.array([0.50, 0.42, 0.34, 0.28, 0.22])       # masses [kg]
J = m * L**2 / 12                                  # slender rod inertia about COM [kg m^2]

g = 9.81

# Joint friction parameters
b_viscous = np.array([0.025, 0.020, 0.015, 0.012, 0.010])  # [Nms/rad]
M_coulomb = np.array([0.030, 0.025, 0.020, 0.015, 0.012])  # [Nm]
v_eps = 0.02                                               # smoothing speed [rad/s]

# Simulation time
omega = 1.5
t_begin = 0.0
t_end = 2 * np.pi / omega
Ts = 0.01
t = np.arange(t_begin, t_end + Ts, Ts)

## Voorgeschreven beweging

Voor inverse dynamica is de beweging gekend. Hieronder kiezen we een vloeiende periodieke beweging voor de vijf relatieve gewrichtshoeken `q_i`. De absolute hoek van stang `i` is:

\[
\phi_i = q_1 + q_2 + \dots + q_i
\]

In [ ]:
phase = np.array([0.0, 0.8, 1.6, 2.2, 2.9])
offset = np.deg2rad([35, -25, 20, -15, 10])
amplitude = np.deg2rad([25, 18, 15, 12, 10])
freq_factor = np.array([1, 1, 2, 2, 3])

q = offset[:, None] + amplitude[:, None] * np.sin(freq_factor[:, None] * omega * t + phase[:, None])
dq = amplitude[:, None] * freq_factor[:, None] * omega * np.cos(freq_factor[:, None] * omega * t + phase[:, None])
ddq = -amplitude[:, None] * (freq_factor[:, None] * omega)**2 * np.sin(freq_factor[:, None] * omega * t + phase[:, None])

phi = np.cumsum(q, axis=0)
dphi = np.cumsum(dq, axis=0)
ddphi = np.cumsum(ddq, axis=0)

## Kinematica van de massacentra

De versnellingen worden analytisch berekend uit de absolute hoek, hoeksnelheid en hoekversnelling van elke stang. Voor een vector `r` vanaf het gewricht naar het massacentrum geldt:

\[
\vec a_G = \vec a_O + \alpha \times \vec r - \omega^2 \vec r
\]

In [ ]:
def unit_vectors(angle):
    return np.stack((np.cos(angle), np.sin(angle)), axis=-1)


def perp(v):
    return np.stack((-v[..., 1], v[..., 0]), axis=-1)


def compute_kinematics(phi, dphi, ddphi, L, c):
    n_links, n_steps = phi.shape
    joint_pos = np.zeros((n_links + 1, n_steps, 2))
    joint_vel = np.zeros((n_links + 1, n_steps, 2))
    joint_acc = np.zeros((n_links + 1, n_steps, 2))
    com_pos = np.zeros((n_links, n_steps, 2))
    com_vel = np.zeros((n_links, n_steps, 2))
    com_acc = np.zeros((n_links, n_steps, 2))

    for i in range(n_links):
        e = unit_vectors(phi[i])
        r_end = L[i] * e
        r_com = c[i] * e

        com_pos[i] = joint_pos[i] + r_com
        com_vel[i] = joint_vel[i] + dphi[i, :, None] * perp(r_com)
        com_acc[i] = joint_acc[i] + ddphi[i, :, None] * perp(r_com) - dphi[i, :, None]**2 * r_com

        joint_pos[i + 1] = joint_pos[i] + r_end
        joint_vel[i + 1] = joint_vel[i] + dphi[i, :, None] * perp(r_end)
        joint_acc[i + 1] = joint_acc[i] + ddphi[i, :, None] * perp(r_end) - dphi[i, :, None]**2 * r_end

    return joint_pos, joint_vel, joint_acc, com_pos, com_vel, com_acc

joint_pos, joint_vel, joint_acc, com_pos, com_vel, com_acc = compute_kinematics(phi, dphi, ddphi, L, c)
com_speed = np.linalg.norm(com_vel, axis=2)

In [ ]:
# Plot one configuration and the path of the end-effector
k_plot = len(t) // 5

plt.figure(figsize=(6, 5))
plt.plot(joint_pos[:, k_plot, 0], joint_pos[:, k_plot, 1], "-o", label="links")
plt.plot(com_pos[:, k_plot, 0], com_pos[:, k_plot, 1], "x", label="centers of mass")
plt.plot(joint_pos[-1, :, 0], joint_pos[-1, :, 1], "--", alpha=0.5, label="end path")
plt.axis("equal")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Five-link mechanism")
plt.legend()
plt.show()

## Inverse dynamica met zwaartekracht en wrijving

De inverse dynamica wordt hier met een achterwaartse Newton-Euler-recursie berekend. Voor elke stang wordt de belasting van alle downstream-stangen mee naar het vorige gewricht genomen.

Voor een stang `i` gebruiken we:

\[
\vec F_i = m_i(\vec a_{G_i} - \vec g) + \vec F_{i+1}
\]

\[
N_i = J_i\alpha_i + \vec r_{G_i}\times m_i(\vec a_{G_i}-\vec g) + N_{i+1} + \vec r_{E_i}\times \vec F_{i+1}
\]

Het gevraagde motorkoppel is dan:

\[
M_i = N_i + M_{f,i}
\]

waarbij `M_f,i = 0` als wrijving wordt uitgeschakeld.

In [ ]:
def cross2(a, b):
    return a[..., 0] * b[..., 1] - a[..., 1] * b[..., 0]


def friction_torque(dq, b_viscous, M_coulomb, v_eps):
    return b_viscous[:, None] * dq + M_coulomb[:, None] * np.tanh(dq / v_eps)


def inverse_dynamics_serial(phi, dphi, ddphi, dq, L, c, m, J, com_acc,
                            include_gravity=True, include_friction=True):
    n_links, n_steps = phi.shape
    torque = np.zeros((n_links, n_steps))
    joint_force = np.zeros((n_links, n_steps, 2))
    gravity = np.array([0.0, -g]) if include_gravity else np.array([0.0, 0.0])
    tau_friction = friction_torque(dq, b_viscous, M_coulomb, v_eps) if include_friction else np.zeros_like(dq)

    for k in range(n_steps):
        force_child = np.zeros(2)
        moment_child = 0.0

        for i in reversed(range(n_links)):
            e = np.array([np.cos(phi[i, k]), np.sin(phi[i, k])])
            r_com = c[i] * e
            r_end = L[i] * e

            inertia_force = m[i] * (com_acc[i, k] - gravity)
            force_total = inertia_force + force_child
            moment_total = (
                J[i] * ddphi[i, k]
                + cross2(r_com, inertia_force)
                + moment_child
                + cross2(r_end, force_child)
            )

            torque[i, k] = moment_total + tau_friction[i, k]
            joint_force[i, k] = force_total

            force_child = force_total
            moment_child = moment_total

    return torque, joint_force, tau_friction

# Three variants make the effects visible:
tau_inertia, F_inertia, tau_fric_zero = inverse_dynamics_serial(
    phi, dphi, ddphi, dq, L, c, m, J, com_acc,
    include_gravity=False, include_friction=False,
)

tau_gravity, F_gravity, _ = inverse_dynamics_serial(
    phi, dphi, ddphi, dq, L, c, m, J, com_acc,
    include_gravity=True, include_friction=False,
)

tau_total, F_total, tau_friction = inverse_dynamics_serial(
    phi, dphi, ddphi, dq, L, c, m, J, com_acc,
    include_gravity=True, include_friction=True,
)

## Krachten en koppels

De grafieken tonen hoe zwaartekracht en wrijving het gevraagde gewrichtskoppel verhogen. De gewrichtskrachten zijn de krachten die het vorige gewricht op de downstream-keten moet uitoefenen.

In [ ]:
fig_tau, ax_tau = plt.subplots(nrows=5, ncols=1, constrained_layout=True, figsize=(9, 10))
fig_tau.suptitle("Joint torques: inertia, gravity and friction")

for i, axis in enumerate(ax_tau):
    axis.plot(t, tau_inertia[i], label="inertia only")
    axis.plot(t, tau_gravity[i], label="with gravity")
    axis.plot(t, tau_total[i], label="with gravity + friction")
    axis.set_ylabel(f"M{i+1} [Nm]")
    axis.legend(loc="upper right")

ax_tau[-1].set_xlabel("t [s]")
plt.show()

In [ ]:
fig_force, ax_force = plt.subplots(nrows=5, ncols=1, constrained_layout=True, figsize=(9, 10))
fig_force.suptitle("Joint force magnitudes with gravity and friction")

F_total_magnitude = np.linalg.norm(F_total, axis=2)
for i, axis in enumerate(ax_force):
    axis.plot(t, F_total_magnitude[i], label=f"joint {i+1}")
    axis.set_ylabel(f"|F{i+1}| [N]")
    axis.legend(loc="upper right")

ax_force[-1].set_xlabel("t [s]")
plt.show()

## Vermogen en motorkeuze

Het mechanisch vermogen per gewricht is:

\[
P_i = M_i\dot q_i
\]

Voor motorkeuze zijn vooral piekkoppel, RMS-koppel, piekvermogen en RMS-vermogen relevant. Een positieve waarde betekent dat de motor energie levert; een negatieve waarde betekent remmen of regeneratie.

In [ ]:
power_total = tau_total * dq
power_positive = np.maximum(power_total, 0.0)
power_negative = np.minimum(power_total, 0.0)

M_peak = np.max(np.abs(tau_total), axis=1)
M_rms = np.sqrt(np.mean(tau_total**2, axis=1))
P_peak = np.max(power_positive, axis=1)
P_rms = np.sqrt(np.mean(power_positive**2, axis=1))
P_regen_peak = -np.min(power_negative, axis=1)

service_factor = 1.5
print("Motor selection per joint")
print("joint | peak torque [Nm] | RMS torque [Nm] | peak power [W] | RMS power [W] | peak braking [W]")
for i in range(n_links):
    print(
        f"{i+1:>5} | {M_peak[i]:>16.4g} | {M_rms[i]:>15.4g} | "
        f"{P_peak[i]:>14.4g} | {P_rms[i]:>13.4g} | {P_regen_peak[i]:>15.4g}"
    )

print()
print(f"Using a service factor of {service_factor:.2f}:")
for i in range(n_links):
    print(
        f"Joint {i+1}: motor peak torque >= {service_factor*M_peak[i]:.4g} Nm, "
        f"continuous torque >= {service_factor*M_rms[i]:.4g} Nm, "
        f"peak power >= {service_factor*P_peak[i]:.4g} W"
    )

In [ ]:
fig_power, ax_power = plt.subplots(nrows=5, ncols=1, constrained_layout=True, figsize=(9, 10))
fig_power.suptitle("Joint powers with gravity and friction")

for i, axis in enumerate(ax_power):
    axis.plot(t, power_total[i], label=f"P{i+1}")
    axis.fill_between(t, 0, power_positive[i], alpha=0.25)
    axis.fill_between(t, 0, power_negative[i], alpha=0.25)
    axis.set_ylabel(f"P{i+1} [W]")
    axis.legend(loc="upper right")

ax_power[-1].set_xlabel("t [s]")
plt.show()

## Controle van het energiebalansprincipe

Als de zwaartekracht en wrijving correct zijn toegevoegd, moet het motorvermogen overeenkomen met:

\[
\frac{dE_{kin}}{dt} + \frac{dE_{pot}}{dt} + P_{wrijving}
\]

Deze controle is gevoeliger dan een gewone plot van koppels, omdat verkeerde tekens in zwaartekracht of wrijving snel zichtbaar worden.

In [ ]:
kinetic_energy = 0.5 * np.sum(m[:, None] * np.sum(com_vel**2, axis=2), axis=0) + 0.5 * np.sum(J[:, None] * dphi**2, axis=0)
potential_energy = np.sum(m[:, None] * g * com_pos[:, :, 1], axis=0)

P_kin = np.sum(m[:, None] * np.sum(com_vel * com_acc, axis=2), axis=0) + np.sum(J[:, None] * dphi * ddphi, axis=0)
P_pot = np.sum(m[:, None] * g * com_vel[:, :, 1], axis=0)
P_friction = np.sum(tau_friction * dq, axis=0)
P_motor_total = np.sum(power_total, axis=0)
P_energy_check = P_kin + P_pot + P_friction

plt.figure(figsize=(9, 4))
plt.plot(t, P_motor_total, label="total motor power")
plt.plot(t, P_energy_check, "--", label=r"$dE_{kin}/dt + dE_{pot}/dt + P_{friction}$")
plt.xlabel("t [s]")
plt.ylabel("Power [W]")
plt.title("Energy balance check")
plt.legend()
plt.show()

plt.figure(figsize=(9, 3))
plt.plot(t, np.abs(P_motor_total - P_energy_check))
plt.xlabel("t [s]")
plt.ylabel("absolute error [W]")
plt.title("Energy balance residual")
plt.show()

print(f"Maximum energy-balance residual: {np.max(np.abs(P_motor_total - P_energy_check)):.4e} W")

## Besluit

Deze vereenvoudigde vijfstangenanalyse is bedoeld als controleerbare demonstratie van inverse dynamica met extra fysische effecten:

- zwaartekracht komt binnen als `m(a_G - g)`;
- wrijving komt rechtstreeks bij het gevraagde motorkoppel;
- gewrichtskrachten volgen uit de achterwaartse krachtrecursie;
- vermogens volgen uit `P = M dq`;
- de energiebalans controleert of de tekens van zwaartekracht en wrijving logisch zijn.

Voor het volledige ruitenwissermechanisme kan dezelfde fysica worden toegevoegd, maar door de gesloten lussen wordt de matrix veel groter en minder geschikt als didactisch voorbeeld.